# Forecasting Ireland's cloud cover with EUMDAC and a CNN–LSTM

**Assignment question:** Can the four most recent satellite cloud masks predict the fraction of Ireland covered by cloud one hour later?

This notebook downloads **EO:EUM:DAT:MSG:CLM**, decodes the original GRIB product, creates a fixed geographical grid and land boundary, constructs samples, trains a PyTorch CNN–LSTM, compares it with persistence and a training-mean baseline, and saves reproducible outputs. Code comments explain both the operations and their purpose.

**Forecast definition:** images at `t−45, t−30, t−15, t` minutes → regional cloud fraction at `t+60`. Cloud and validity are separate image channels; UTC calendar features are additional covariates. “Ireland” defaults to the **Republic of Ireland**; supply a whole-island polygon if required. This is a regional fraction forecast, not a pixelwise cloud map or a precipitation forecast.

**Before Run All:** install the environment below; edit the configuration cell (especially dates and output path). For real data, supply EUMETSAT credentials through environment variables or the hidden prompts. No credentials are saved. A clearly separated `synthetic` mode exercises the learning pipeline without credentials or satellite downloads. Its results are software checks, not scientific evidence.

**Validation boundary:** the supplied notebook was checked with synthetic data. A live authenticated EUMETSAT download and real-file decoding must still be verified in your environment. No real-data accuracy is claimed.


## 1. Install once and select the environment's Jupyter kernel

Use Python 3.11. A conda environment is recommended because the GRIB and geographical packages have native dependencies. Run these commands in a terminal, then open this notebook in JupyterLab:

```bash
conda create -n ireland-cloud -c conda-forge python=3.11 jupyterlab ipykernel numpy pandas matplotlib pytorch satpy pyresample eccodes python-eccodes geopandas shapely pyproj requests tqdm pip
conda activate ireland-cloud
python -m pip install "eumdac>=2.2.3,<4"
python -m ipykernel install --user --name ireland-cloud --display-name "Python (Ireland cloud)"
jupyter lab
```

If using pip instead, install `numpy pandas matplotlib torch satpy pyresample eccodes geopandas shapely pyproj requests tqdm eumdac jupyterlab ipykernel`. If ecCodes cannot load its native library, use the conda route. GPU acceleration is optional; install a compatible PyTorch build using its official installation instructions. Restart the kernel after installation.

For synthetic mode only, `numpy pandas matplotlib torch jupyterlab ipykernel` are enough. Package versions are recorded at run time; the environment recipe is not an exact lockfile. Do not install dependencies in the middle of a training run.


In [1]:
# General-purpose and ML imports; satellite/GIS imports are delayed until real mode.
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import os, json, hashlib, shutil, zipfile, time, random, getpass, gc
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from IPython.display import display

# ---------------- USER SETTINGS: EDIT THIS BLOCK ----------------
MODE = 'real'                      # 'real' for the assignment; 'synthetic' for a smoke test
ROOT = Path.cwd() / 'ireland_cloud_work'  # CHANGE if you want a different writable directory
START_UTC = '2024-06-01T00:00:00Z'  # Inclusive observation-time boundary
END_UTC = '2024-06-15T00:00:00Z'    # Exclusive boundary; start with a short pilot
MAX_PRODUCTS = 2000               # Raise deliberately for longer experiments
BOUNDARY_PATH = None              # Optional Path to YOUR GeoJSON/GPKG/shapefile
REGION_NAME = 'Republic of Ireland' # Change when supplying a whole-island polygon
# A custom file must contain ONLY the land region to use (all its features are unioned).
# It must declare its CRS. Leave None to download Natural Earth's Ireland polygon.
COLLECTION_ID = 'EO:EUM:DAT:MSG:CLM'
LOOKBACK, STEP_MINUTES, HORIZON_MINUTES = 4, 15, 60
MIN_TARGET_COVERAGE = 0.98         # Proposed QC rule, not a product specification
MIN_INPUT_COVERAGE = 0.90         # Valid fraction of surrounding image for each input
INPUT_SIZE = 64                   # Spatial covariates: 64 x 64; try 128 after the pilot
FINE_SIZE = 400                   # 1200-km square / 400 = 3-km target grid cells
HALF_WIDTH_M = 600_000
RADIUS_M = 12_000                 # Maximum nearest-neighbour search distance
TRAIN_SHARE, VALID_SHARE = 0.70, 0.15
BATCH_SIZE, EPOCHS, PATIENCE = 32, 30, 5
LEARNING_RATE, SEED = 1e-3, 42
# ---------------------------------------------------------------
assert MODE in {'real', 'synthetic'}
assert LOOKBACK >= 1 and HORIZON_MINUTES > 0 and STEP_MINUTES > 0
assert HORIZON_MINUTES % STEP_MINUTES == 0
assert 0 < TRAIN_SHARE < TRAIN_SHARE + VALID_SHARE < 1
assert INPUT_SIZE >= 16 and EPOCHS > 0 and PATIENCE > 0
start, end = pd.Timestamp(START_UTC), pd.Timestamp(END_UTC)
assert start.tzinfo is not None and end.tzinfo is not None and start < end
start, end = start.tz_convert('UTC'), end.tz_convert('UTC')
ROOT = ROOT.expanduser().resolve()
RAW, GEO, CACHE = [ROOT / name for name in ('raw', 'geography', 'processed')]
for folder in (RAW, GEO, CACHE):
    folder.mkdir(parents=True, exist_ok=True)
# Each execution gets its own result directory; raw downloads can be reused.
RUN = ROOT / 'runs' / (MODE + '_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
RUN.mkdir(parents=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Few CPU threads avoid overhead on these small teaching examples.
torch.set_num_threads(min(4, os.cpu_count() or 1))
print('Mode:', MODE, '| Device:', DEVICE, '| Results:', RUN)


Mode: real | Device: cpu | Results: /home/evanoreilly/ai_code/AIfW-Presentation-2/ireland_cloud_work/runs/real_20260928T150220801235Z


/home/evanoreilly/ai_code/.venv/lib64/python3.13/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


## 2. Download original products with EUMDAC

Create an EUMETSAT account, accept any required collection terms, and get a consumer key and secret from the [API key page](https://api.eumetsat.int/api-key/). Set `EUMETSAT_CONSUMER_KEY` and `EUMETSAT_CONSUMER_SECRET` in your environment, or enter them in the hidden prompts below. Do not put secrets in notebook source, screenshots, or shared outputs.

The query is made one day at a time to keep search sizes manageable. Products are downloaded serially, with retries, a maximum-count guard, and atomic `.part` writes. Existing nonempty downloads are reused; recorded SHA-256 digests detect changed or truncated cached files. Original filenames are retained because Satpy uses them to identify this product. Bounding-box search is unnecessary for a full-disk product: the actual regional crop happens locally.

The dates are a 14-day historical pilot (about 1,344 scheduled observations before gaps). Multi-week or multi-month experiments will be more informative and may require substantial disk space and preprocessing time. Query availability is checked rather than assumed. Do not silently substitute another collection: its format and class meanings may differ.


In [2]:
def sha256_file(path):
    """Hash in blocks so even large raw files need little extra memory."""
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def write_json(path, value):
    """Use a temporary file so an interrupted write does not look complete."""
    temp = Path(str(path) + '.part')
    temp.write_text(json.dumps(value, indent=2, default=str), encoding='utf-8')
    temp.replace(path)

def unpack_product(archive, destination):
    """Extract ordinary ZIP members safely, preserving their original names."""
    destination = destination.resolve()
    if not zipfile.is_zipfile(archive):
        # Some products are delivered directly, rather than inside a ZIP.
        return [archive]
    files = []
    with zipfile.ZipFile(archive) as bundle:
        for member in bundle.infolist():
            parts = PurePosixPath(member.filename.replace('\\', '/')).parts
            if not parts or '..' in parts or any(':' in p for p in parts):
                raise ValueError('Unsafe archive member name')
            target = (destination / Path(*parts)).resolve()
            if not target.is_relative_to(destination):
                raise ValueError('Archive member escapes extraction directory')
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            # Reading fully also makes zipfile check the member's CRC.
            temp = Path(str(target) + '.part')
            with bundle.open(member) as src, open(temp, 'wb') as dst:
                shutil.copyfileobj(src, dst)
            temp.replace(target)
            files.append(target)
    return files

raw_records = []
if MODE == 'real':
    import eumdac
    key = os.environ.get('EUMETSAT_CONSUMER_KEY') or getpass.getpass('EUMETSAT consumer key: ')
    secret = os.environ.get('EUMETSAT_CONSUMER_SECRET') or getpass.getpass('EUMETSAT consumer secret: ')
    if not key or not secret:
        raise ValueError('Supply both EUMETSAT credentials to run real mode.')
    token = eumdac.AccessToken((key, secret))
    store = eumdac.DataStore(token)
    collection = store.get_collection(COLLECTION_ID)
    del key, secret  # The access-token object still needs credentials to refresh itself.

    # Deduplicate IDs because adjacent inclusive search intervals may share endpoints.
    products = {}
    left = start
    while left < end:
        right = min(left + pd.Timedelta(days=1), end)
        for product in collection.search(dtstart=left.to_pydatetime(), dtend=right.to_pydatetime()):
            products[str(product)] = product
            if len(products) > MAX_PRODUCTS:
                raise RuntimeError('Query exceeds MAX_PRODUCTS. Shorten dates or raise the guard.')
        left = right
    if not products:
        raise RuntimeError('No products found. Check collection access and date availability.')
    print('Unique catalogue products:', len(products))

    for number, (product_id, product) in enumerate(sorted(products.items()), 1):
        # An ID hash avoids filesystem-incompatible characters without renaming the data file.
        product_dir = RAW / hashlib.sha256(product_id.encode()).hexdigest()[:20]
        product_dir.mkdir(exist_ok=True)
        receipt_path = product_dir / 'receipt.json'
        receipt = json.loads(receipt_path.read_text()) if receipt_path.exists() else None
        archive = product_dir / receipt['filename'] if receipt else None
        reusable = (archive is not None and archive.exists() and archive.stat().st_size > 0
                    and sha256_file(archive) == receipt['sha256'])
        if not reusable:
            for attempt in range(3):
                try:
                    with product.open() as src:
                        # EUMDAC exposes the server's delivered filename on the stream.
                        filename = Path(str(src.name).replace('\\', '/')).name
                        if filename in {'', '.', '..'}:
                            raise ValueError('Product stream has no usable filename')
                        archive = product_dir / filename
                        temp = Path(str(archive) + '.part')
                        with open(temp, 'wb') as dst:
                            shutil.copyfileobj(src, dst)
                    if temp.stat().st_size == 0:
                        raise IOError('Empty product download')
                    temp.replace(archive)
                    receipt = {'product_id': product_id, 'filename': filename,
                               'sha256': sha256_file(archive), 'bytes': archive.stat().st_size}
                    write_json(receipt_path, receipt)
                    break
                except Exception:
                    if attempt == 2:
                        raise
                    time.sleep(2 ** (attempt + 1))
        files = unpack_product(archive, product_dir / 'extracted')
        # GRIB magic works even for original products without a filename extension.
        grib_files = []
        for file in files:
            with open(file, 'rb') as stream:
                if stream.read(4) == b'GRIB':
                    grib_files.append(file)
        if not grib_files:
            raise RuntimeError(f'No original GRIB found in {archive}; inspect the delivery format.')
        for file in grib_files:
            raw_records.append({'product_id': product_id, 'file': str(file),
                                'sha256': sha256_file(file)})
        if number == 1 or number % 50 == 0 or number == len(products):
            print(f'Downloaded/reused {number}/{len(products)} products')
    write_json(RUN / 'download_manifest.json', raw_records)
else:
    print('Synthetic mode: no credentials, downloads, or real-data preprocessing.')


Unique catalogue products: 1345


RuntimeError: No original GRIB found in /home/evanoreilly/ai_code/AIfW-Presentation-2/ireland_cloud_work/raw/0b1f409bd8f8e8ecd22d/MSG3-SEVI-MSGCLMK-0100-0100-20240601000000.000000000Z-NA.zip; inspect the delivery format.

## 3. Define a fixed boundary and equal-area grids

The default boundary is Natural Earth's 1:10-million-scale Ireland country polygon. It is a convenient, generalized coastline rather than a survey boundary. For the whole island, supply a file containing the Republic and Northern Ireland land polygons; do not include the whole UK. All features in a custom file are combined.

Both grids use a Lambert azimuthal equal-area projection centred on Ireland. The 400 × 400 target grid has 3-km cells; the surrounding 64 × 64 input grid is coarser. A fixed **cell-centre-in-polygon** mask defines the target. Each selected cell has equal area, so the target is cloudy valid cells divided by valid selected cells. Coastline cells are approximated by their centres; resampling to 3 km does not create finer satellite information.

Cloud values are categorical: 0 = clear water, 1 = clear land, 2 = cloud, 3 = no data. Other missing representations, including 255 and NaN, are invalid. Never average category numbers. We use nearest-neighbour resampling and preserve invalid cells explicitly, avoiding interpolation of missing observations into cloud categories.


In [ ]:
PROJECTION = {'proj': 'laea', 'lat_0': 53.5, 'lon_0': -8.0, 'datum': 'WGS84', 'units': 'm'}
EXTENT = (-HALF_WIDTH_M, -HALF_WIDTH_M, HALF_WIDTH_M, HALF_WIDTH_M)
# Matplotlib orders bounds differently from AreaDefinition/Shapely.
PLOT_EXTENT = (EXTENT[0], EXTENT[2], EXTENT[1], EXTENT[3])
if MODE == 'real':
    import requests
    import geopandas as gpd
    from shapely import contains_xy, box
    from pyresample.geometry import AreaDefinition
    from satpy import Scene

    if BOUNDARY_PATH is None:
        boundary_archive = GEO / 'ne_10m_admin_0_countries.zip'
        boundary_url = 'https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_0_countries.zip'
        if not boundary_archive.exists():
            response = requests.get(boundary_url, timeout=120)
            response.raise_for_status()
            temp = boundary_archive.with_suffix('.part')
            temp.write_bytes(response.content)
            if not zipfile.is_zipfile(temp):
                raise ValueError('Boundary download is not a ZIP archive')
            temp.replace(boundary_archive)
        countries = gpd.read_file(boundary_archive)
        boundary = countries.loc[countries['ADMIN'].eq('Ireland')].copy()
        boundary_source = boundary_url
    else:
        boundary = gpd.read_file(Path(BOUNDARY_PATH).expanduser())
        boundary_source = str(Path(BOUNDARY_PATH).expanduser().resolve())
    if boundary.empty or boundary.crs is None:
        raise ValueError('Boundary is empty or has no declared CRS')
    if boundary.geometry.isna().any() or not boundary.geometry.is_valid.all():
        raise ValueError('Repair missing/invalid boundary geometry before use')
    boundary = boundary.to_crs(PROJECTION)
    polygon = boundary.geometry.union_all()
    if not box(*EXTENT).covers(polygon):
        raise ValueError('Boundary extends outside the configured grid; check region and extent')
    boundary.to_crs('EPSG:4326').to_file(RUN / 'target_boundary.geojson', driver='GeoJSON')
    boundary_digest = hashlib.sha256(polygon.wkb).hexdigest()

    def area(size, name):
        return AreaDefinition(name, name, 'ireland_laea', PROJECTION, size, size, EXTENT)
    fine_area = area(FINE_SIZE, 'ireland_target')
    input_area = area(INPUT_SIZE, 'ireland_context')
    x, y = fine_area.get_proj_vectors()  # Columns west→east; rows north→south.
    xx, yy = np.meshgrid(x, y)
    ireland_mask = contains_xy(polygon, xx, yy)
    if ireland_mask.sum() == 0:
        raise ValueError('Boundary contains no target-grid cell centres')
    np.save(RUN / 'ireland_mask.npy', ireland_mask)
    cell_area_km2 = ((2 * HALF_WIDTH_M / FINE_SIZE) / 1000) ** 2
    print('Approximate masked land area (km²):', ireland_mask.sum() * cell_area_km2)
    print('Polygon land area (km²):', polygon.area / 1e6)
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.imshow(ireland_mask, extent=PLOT_EXTENT, origin='upper', cmap='Blues')
    boundary.boundary.plot(ax=ax, color='black', linewidth=0.7)
    ax.set(title=f'{REGION_NAME}: fixed target mask', xlabel='Easting (m)', ylabel='Northing (m)')
    fig.savefig(RUN / 'boundary_check.png', dpi=150, bbox_inches='tight')
    plt.show()


## 4. Decode, resample, quality-check, and cache each frame

Satpy's `seviri_l2_grib` reader loads `cloud_mask` from an original GRIB file. If its filename is unsupported, stop and inspect the original delivery; do not rename arbitrary formats to make them appear compatible. The observation time comes from the reader's sensing metadata, never the download time. Exact 15-minute spacing is checked later; times are not rounded and missing frames are not filled.

To keep missing cells from being silently replaced by nearby valid observations, missing values are converted to category 3 before resampling and missing pixels are not masked out of the nearest-neighbour search. All native categories are validated first. The 12-km search radius is a pilot choice: inspect coverage and the preview before committing to a long run.

The target is rejected below 98% valid land coverage. Input sequences require at least 90% coverage in every surrounding-region image, and a valid last-observed regional fraction for the persistence comparison. This selects a common evaluation population and may exclude difficult missing-data periods; report that exclusion. Caches include the raw content hash, geography, settings, and reader versions so changed preprocessing does not reuse stale frames.


In [ ]:
def summarize_categories(values, region_mask, threshold):
    """Return a fraction and coverage; missing pixels never count as clear sky."""
    valid = np.isin(values, [0, 1, 2])
    selected_valid = valid & region_mask
    coverage = float(selected_valid.sum() / region_mask.sum())
    fraction = (float(((values == 2) & selected_valid).sum() / selected_valid.sum())
                if selected_valid.any() and coverage >= threshold else np.nan)
    return fraction, coverage

if MODE == 'real':
    preprocess_config = dict(schema=1, projection=PROJECTION, extent=EXTENT,
        input_size=INPUT_SIZE, fine_size=FINE_SIZE, radius_m=RADIUS_M,
        min_target_coverage=MIN_TARGET_COVERAGE, boundary_sha256=boundary_digest,
        versions={p: metadata.version(p) for p in ('satpy', 'pyresample', 'eccodes')})
    config_digest = hashlib.sha256(json.dumps(preprocess_config, sort_keys=True).encode()).hexdigest()
    processed_records, cache_paths = [], []
    preview_values = None
    for number, record in enumerate(raw_records, 1):
        cache_key = hashlib.sha256((record['sha256'] + config_digest).encode()).hexdigest()
        cached = CACHE / f'{cache_key}.npz'
        if not cached.exists():
            scene = Scene(reader='seviri_l2_grib', filenames=[record['file']])
            scene.load(['cloud_mask'])
            native = scene['cloud_mask']
            observation_time = pd.Timestamp(native.attrs['start_time'])
            observation_time = (observation_time.tz_localize('UTC') if observation_time.tzinfo is None
                                else observation_time.tz_convert('UTC'))
            # Only compute unique values for validation; the grid remains geolocated in Satpy.
            values = np.asarray(native.values)
            finite_values = np.unique(values[np.isfinite(values)])
            if not np.isin(finite_values, [0, 1, 2, 3, 255]).all():
                raise ValueError(f'Unexpected CLM categories {finite_values}; inspect {record["file"]}')
            scene['cloud_mask'] = native.where(native.isin([0, 1, 2]), 3).astype('uint8')
            arrays = []
            for grid in (input_area, fine_area):
                local = scene.resample(grid, resampler='nearest', radius_of_influence=RADIUS_M,
                                       fill_value=3, mask_area=False)
                arrays.append(np.asarray(local['cloud_mask'].values))
            coarse, fine = arrays
            fraction, coverage = summarize_categories(fine, ireland_mask, MIN_TARGET_COVERAGE)
            valid = np.isin(coarse, [0, 1, 2])
            frame = np.stack([coarse == 2, valid]).astype('uint8')
            # Write a complete cache before atomically replacing its target.
            temp = Path(str(cached) + '.part')
            with open(temp, 'wb') as stream:
                np.savez_compressed(stream, frame=frame, fraction=np.float32(fraction),
                    target_coverage=np.float32(coverage), input_coverage=np.float32(valid.mean()),
                    time=np.array(observation_time.isoformat()))
            temp.replace(cached)
            if preview_values is None:
                preview_values = fine.copy()
            del scene, native, local, values, arrays, coarse, fine
            gc.collect()
        with np.load(cached, allow_pickle=False) as saved:
            stamp = pd.Timestamp(str(saved['time'].item()))
            # EUMDAC search endpoints can be inclusive; enforce [start, end) here.
            if not start <= stamp < end:
                continue
            processed_records.append({**record, 'time': stamp,
                'fraction': float(saved['fraction']), 'target_coverage': float(saved['target_coverage']),
                'input_coverage': float(saved['input_coverage']), 'cache': str(cached)})
        if number == 1 or number % 50 == 0:
            print(f'Processed/reused {number}/{len(raw_records)} files')
    if not processed_records:
        raise RuntimeError('No decoded observations fall inside the requested period')
    table = pd.DataFrame(processed_records).sort_values(['time', 'sha256']).reset_index(drop=True)
    # Identical duplicate deliveries are harmless. Conflicting observations require a decision.
    conflicts = table.groupby('time')['sha256'].nunique()
    if (conflicts > 1).any():
        raise ValueError('Different files share an observation time; inspect platform/reprocessing versions')
    table = table.drop_duplicates('time').reset_index(drop=True)
    frames = []
    for path in table['cache']:
        with np.load(path, allow_pickle=False) as saved:
            frames.append(saved['frame'])
    frames = np.stack(frames)
    times = pd.DatetimeIndex(table['time'])
    fractions = table['fraction'].to_numpy(dtype='float32')
    input_coverage = table['input_coverage'].to_numpy(dtype='float32')
    table.to_csv(RUN / 'frame_manifest.csv', index=False)
    write_json(RUN / 'preprocessing.json', {**preprocess_config, 'boundary_source': boundary_source})
    if preview_values is not None:
        fig, ax = plt.subplots(figsize=(7, 6))
        ax.imshow(np.ma.masked_where(preview_values == 3, preview_values), extent=PLOT_EXTENT,
                  origin='upper', vmin=0, vmax=2, cmap='viridis')
        boundary.boundary.plot(ax=ax, color='red', linewidth=0.7)
        ax.set_title('First newly decoded CLM: 0 water, 1 land, 2 cloud; missing masked')
        fig.savefig(RUN / 'decoded_preview.png', dpi=150, bbox_inches='tight')
        plt.show()


## 5. Optional synthetic smoke test and prepared-data checks

In `synthetic` mode the following cell makes moving artificial cloud shapes and an artificial elliptical land mask. It bypasses EUMDAC, GRIB, and real geography. It exercises tensor shapes, target generation, chronological splitting, training, baselines, and saving. Keep these results separate from the assignment's real-data results.

In real mode, the preceding cells supply the same array interface. `frames` is `[time, 2, height, width]`, with cloud and valid channels; `fractions` is the fine-grid regional response. The `.npz` output can be reused for later ML experiments without decoding again. In this teaching implementation all coarse frames are in memory; for many months use a disk-backed dataset or memory mapping.


In [ ]:
if MODE == 'synthetic':
    times = pd.date_range('2024-06-01', periods=384, freq=f'{STEP_MINUTES}min', tz='UTC')
    yy, xx = np.mgrid[-1:1:complex(INPUT_SIZE), -1:1:complex(INPUT_SIZE)]
    artificial_land = ((xx / 0.35) ** 2 + (yy / 0.6) ** 2) < 1
    frames, fractions = [], []
    for i in range(len(times)):
        field = np.sin(4 * xx + i * 0.11) + np.cos(5 * yy - i * 0.07)
        cloudy = field > 0.15
        valid = np.ones_like(cloudy, dtype=bool)
        frames.append(np.stack([cloudy, valid]).astype('uint8'))
        fractions.append(float(cloudy[artificial_land].mean()))
    frames = np.stack(frames)
    fractions = np.asarray(fractions, dtype='float32')
    input_coverage = np.ones(len(times), dtype='float32')
    EPOCHS = min(EPOCHS, 2)
    print('SYNTHETIC DATA ONLY: artificial cloud patterns and an artificial land mask')

assert len(times) == len(frames) == len(fractions) == len(input_coverage)
assert times.is_monotonic_increasing and times.is_unique
assert frames.shape[1:] == (2, INPUT_SIZE, INPUT_SIZE)
assert np.isin(frames, [0, 1]).all()
assert (frames[:, 0] <= frames[:, 1]).all()  # Cloud cannot be set at invalid pixels.
assert np.isfinite(fractions).any()
assert np.all((fractions[np.isfinite(fractions)] >= 0) & (fractions[np.isfinite(fractions)] <= 1))
np.savez_compressed(RUN / 'prepared_data.npz', frames=frames, fractions=fractions,
    times=times.tz_localize(None).to_numpy(dtype='datetime64[s]'), input_coverage=input_coverage)
delta = times[1:] - times[:-1]
print('Frames:', len(times), '| Valid regional fractions:', np.isfinite(fractions).sum())
print('Non-15-minute intervals:', int((delta != pd.Timedelta(minutes=STEP_MINUTES)).sum()))
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
axes[0].imshow(frames[0, 0], vmin=0, vmax=1, cmap='gray')
axes[0].set_title('First cloud channel')
axes[1].imshow(frames[0, 1], vmin=0, vmax=1, cmap='gray')
axes[1].set_title('First validity channel')
axes[2].plot(times, fractions)
axes[2].set(title=f'{MODE}: regional fraction', ylim=(0, 1))
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(RUN / 'prepared_preview.png', dpi=150)
plt.show()


## 6. Build samples within chronological partitions

The first 70% of elapsed time is training, the next 15% validation, and the last 15% test. Each sample's complete span, from its earliest input to its future target, must fit within one partition. This discards boundary-crossing windows, so no observation is shared between partitions. Training batches may subsequently be shuffled because the LSTM state resets for each example.

We require the complete input-to-target span to be on a contiguous 15-minute sequence. This conservative rule also rejects missing intermediate future frames, even though they are not model inputs. The target and persistence value must both meet the regional quality threshold. No future image or observed future fraction becomes a covariate.

Calendar encodings are sine/cosine of UTC time of day and fraction of the actual calendar year (including leap years), evaluated at the **input** times. They have fixed scales, so no train/test normalization is needed.


In [ ]:
def calendar_features(index):
    """Encode circular clocks without an artificial jump at midnight/New Year."""
    day_phase = (index.hour * 3600 + index.minute * 60 + index.second) / 86400
    year_length = np.where(index.is_leap_year, 366, 365)
    year_phase = ((index.dayofyear - 1) + day_phase) / year_length
    return np.column_stack([np.sin(2*np.pi*day_phase), np.cos(2*np.pi*day_phase),
                            np.sin(2*np.pi*year_phase), np.cos(2*np.pi*year_phase)]).astype('float32')

calendar = calendar_features(times)
step = pd.Timedelta(minutes=STEP_MINUTES)
horizon_steps = HORIZON_MINUTES // STEP_MINUTES
timeline_end = times[-1] + step
cut1 = times[0] + (timeline_end - times[0]) * TRAIN_SHARE
cut2 = times[0] + (timeline_end - times[0]) * (TRAIN_SHARE + VALID_SHARE)
partitions = {'train': (times[0], cut1), 'validation': (cut1, cut2), 'test': (cut2, timeline_end)}

def build_samples(index, values, coverage, partitions):
    """Index rows only; the Dataset fetches images later to avoid copying windows."""
    rows, rejected = [], {'gap': 0, 'quality': 0, 'split_boundary': 0}
    for origin in range(LOOKBACK - 1, len(index) - horizon_steps):
        first, target = origin - LOOKBACK + 1, origin + horizon_steps
        span = index[first:target + 1]
        if not ((span[1:] - span[:-1]) == step).all():
            rejected['gap'] += 1
            continue
        if (not np.isfinite(values[target]) or not np.isfinite(values[origin])
                or not np.all(coverage[first:origin + 1] >= MIN_INPUT_COVERAGE)):
            rejected['quality'] += 1
            continue
        split = next((name for name, (left, right) in partitions.items()
                      if index[first] >= left and index[target] < right), None)
        if split is None:
            rejected['split_boundary'] += 1
            continue
        rows.append(dict(first=first, origin=origin, target=target, split=split,
                         origin_time=index[origin], target_time=index[target]))
    return pd.DataFrame(rows, columns=['first','origin','target','split','origin_time','target_time']), rejected

samples, rejected = build_samples(times, fractions, input_coverage, partitions)
print('Rejected candidate samples:', rejected)
print('Retained samples:', samples.groupby('split').size().to_dict())
for name in partitions:
    if (samples['split'] == name).sum() < 10:
        raise RuntimeError(f'Too few {name} samples. Extend dates or inspect gaps/quality; do not random-split.')
samples.to_csv(RUN / 'sample_manifest.csv', index=False)
display(samples.head())

class CloudDataset(Dataset):
    def __init__(self, sample_table):
        self.rows = sample_table.reset_index(drop=True)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, item):
        row = self.rows.iloc[item]
        first, origin, target = int(row['first']), int(row['origin']), int(row['target'])
        # NumPy uint8 frames become floating-point tensors only for the current sample.
        return (torch.from_numpy(frames[first:origin+1].astype('float32')),
                torch.from_numpy(calendar[first:origin+1]),
                torch.tensor(fractions[target], dtype=torch.float32))

datasets = {name: CloudDataset(samples.loc[samples['split'].eq(name)]) for name in partitions}
generator = torch.Generator().manual_seed(SEED)
loaders = {name: DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=(name == 'train'),
            num_workers=0, generator=generator if name == 'train' else None)
           for name, dataset in datasets.items()}
# num_workers=0 works reliably in Windows notebooks and keeps memory use predictable.


## 7. Define the CNN–LSTM

A shared CNN turns each image into 512 spatial features. Four calendar features are appended, then the LSTM summarizes the four-frame sequence. A small regression head predicts a single fraction in [0, 1]. This is a CNN followed by an LSTM, rather than a ConvLSTM layer. The network is deliberately modest for a pilot.

Input shapes: images `[batch, 4, 2, 64, 64]`, calendar `[batch, 4, 4]`, targets `[batch]`. Adaptive pooling keeps the model compatible with changing the configured image size. Mean squared error is a regression loss; the sigmoid output is regional coverage, not the probability that the entire region is cloudy.


In [ ]:
class CloudCNNLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(2, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((4, 4)), nn.Flatten())
        self.lstm = nn.LSTM(input_size=512 + 4, hidden_size=64, batch_first=True)
        self.head = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.1),
                                  nn.Linear(32, 1), nn.Sigmoid())

    def forward(self, images, time_features):
        batch, length, channels, height, width = images.shape
        # Share exactly the same convolutional weights across all input times.
        spatial = self.cnn(images.reshape(batch * length, channels, height, width))
        spatial = spatial.reshape(batch, length, 512)
        sequence = torch.cat([spatial, time_features], dim=-1)
        temporal, _ = self.lstm(sequence)  # Fresh recurrent state on each call.
        return self.head(temporal[:, -1]).squeeze(-1)

model = CloudCNNLSTM().to(DEVICE)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
images, clocks, target = next(iter(loaders['train']))
with torch.no_grad():
    check = model(images.to(DEVICE), clocks.to(DEVICE))
assert check.shape == target.shape
print('Input shape:', tuple(images.shape), '| Prediction shape:', tuple(check.shape))
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))


## 8. Train with validation-based early stopping

Only the training split updates weights. Validation MSE chooses the checkpoint and triggers early stopping; test data are untouched until the next section. Epoch losses are weighted by batch size so the short final batch does not count as much as a full batch. Gradient clipping reduces unstable LSTM updates. Reproducibility is improved by fixed seeds, but results may still vary across hardware or library versions.


In [ ]:
def run_epoch(loader, training):
    model.train(training)  # Also enables/disables dropout.
    total_squared_error, count = 0.0, 0
    with torch.set_grad_enabled(training):
        for images, clocks, target in loader:
            images, clocks, target = images.to(DEVICE), clocks.to(DEVICE), target.to(DEVICE)
            if training:
                optimizer.zero_grad(set_to_none=True)
            predicted = model(images, clocks)
            loss = loss_fn(predicted, target)
            if not torch.isfinite(loss):
                raise RuntimeError('Non-finite loss; inspect data and learning rate')
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            total_squared_error += loss.item() * len(target)
            count += len(target)
    return total_squared_error / count

best_val, stale, best_epoch = float('inf'), 0, None
history = []
checkpoint = RUN / 'best_model.pt'
for epoch in range(1, EPOCHS + 1):
    train_mse = run_epoch(loaders['train'], training=True)
    val_mse = run_epoch(loaders['validation'], training=False)
    history.append({'epoch': epoch, 'train_mse': train_mse, 'validation_mse': val_mse})
    print(f'Epoch {epoch:02d} | train MSE {train_mse:.5f} | validation MSE {val_mse:.5f}')
    if val_mse < best_val:
        best_val, stale, best_epoch = val_mse, 0, epoch
        # Save CPU tensors, making the checkpoint portable to a CPU-only machine.
        torch.save({k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, checkpoint)
    else:
        stale += 1
        if stale >= PATIENCE:
            print('Early stopping: no validation improvement.')
            break

model.load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True))
history = pd.DataFrame(history)
history.to_csv(RUN / 'training_history.csv', index=False)
fig, ax = plt.subplots(figsize=(7, 4))
history.plot(x='epoch', y=['train_mse', 'validation_mse'], ax=ax)
ax.set(ylabel='MSE (fraction²)', title=f'{MODE}: learning curves; selected epoch {best_epoch}')
fig.savefig(RUN / 'learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Evaluate once against simple baselines

**Persistence** predicts that cloud fraction at `t+60` equals the observed fraction at `t`. **Training mean** predicts the mean target of the training samples, without using validation or test labels. All three methods use exactly the same retained test samples. MAE and RMSE are reported in **percentage points** (fraction error × 100), not relative percent error.

MSE skill against persistence is `1 − MSE(model)/MSE(persistence)`; positive is better, zero is equal, negative is worse. It is undefined when persistence has zero error. Closely spaced test windows are correlated, so the sample count is not a count of independent trials. Do not interpret a small difference as statistical significance. For an extended study, use additional held-out periods or time-block bootstrap uncertainty; reserve the test set from architecture tuning.


In [ ]:
model.eval()
predictions = []
with torch.no_grad():
    for images, clocks, _ in loaders['test']:
        predictions.append(model(images.to(DEVICE), clocks.to(DEVICE)).cpu().numpy())
network_predictions = np.concatenate(predictions)
test_rows = datasets['test'].rows
train_target_indices = datasets['train'].rows['target'].to_numpy(dtype=int)
training_mean = float(fractions[train_target_indices].mean())
truth = fractions[test_rows['target'].to_numpy(dtype=int)]
persistence = fractions[test_rows['origin'].to_numpy(dtype=int)]
test_predictions = pd.DataFrame({
    'origin_time_utc': test_rows['origin_time'], 'target_time_utc': test_rows['target_time'],
    'observed_fraction': truth, 'cnn_lstm_fraction': network_predictions,
    'persistence_fraction': persistence, 'training_mean_fraction': training_mean})
test_predictions.to_csv(RUN / 'test_predictions.csv', index=False)

def errors(y, predicted):
    difference = np.asarray(predicted) - np.asarray(y)
    return {'MAE_pp': 100 * float(np.abs(difference).mean()),
            'RMSE_pp': 100 * float(np.sqrt(np.square(difference).mean())),
            'bias_pp': 100 * float(difference.mean()), 'MSE': float(np.square(difference).mean())}

methods = {'CNN-LSTM': network_predictions, 'Persistence': persistence,
           'Training mean': np.full_like(truth, training_mean)}
metrics = pd.DataFrame({name: errors(truth, pred) for name, pred in methods.items()}).T
baseline_mse = metrics.loc['Persistence', 'MSE']
metrics['MSE_skill_vs_persistence'] = (1 - metrics['MSE'] / baseline_mse
                                         if baseline_mse > 0 else np.nan)
metrics['n_test'] = len(truth)
metrics.to_csv(RUN / 'test_metrics.csv', index_label='method')
print('Evaluation mode:', MODE)
display(metrics)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
# Show a short contiguous part of the test period so individual forecasts are readable.
shown = test_predictions.iloc[:192]
for column in ('observed_fraction', 'cnn_lstm_fraction', 'persistence_fraction'):
    axes[0].plot(shown['target_time_utc'], 100 * shown[column], label=column)
axes[0].set(ylabel='Cloud cover (%)', title=f'{MODE}: first test forecasts', ylim=(0, 100))
axes[0].legend(fontsize=8)
axes[1].scatter(100*truth, 100*network_predictions, s=10, alpha=0.4)
axes[1].plot([0, 100], [0, 100], '--', color='black')
axes[1].set(xlabel='Observed coverage (%)', ylabel='CNN–LSTM coverage (%)',
            title=f'{MODE}: held-out predictions', xlim=(0, 100), ylim=(0, 100))
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(RUN / 'test_forecasts.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Save provenance and demonstrate checkpoint reuse

The run directory contains the best weights, prepared arrays, sample timestamps, predictions, metrics, learning curves, settings, and package versions. Real runs also save raw-file receipts, a frame manifest, the boundary, the target mask, and preprocessing settings. Raw products and per-frame caches live outside individual run directories and can be reused. No credential values are included.

Keep this notebook with the weights: the model class is needed to reload a state dictionary. Apply exactly the same boundary, grid, class handling, and calendar features to new observations. A forecast origin denotes an observation time; actual operational availability may be delayed by satellite scanning and product delivery. This pilot does not model that latency.


In [ ]:
versions = {}
for package in ('numpy', 'pandas', 'matplotlib', 'torch', 'eumdac', 'satpy', 'pyresample',
                'eccodes', 'geopandas', 'shapely', 'pyproj'):
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = 'not installed (not needed in synthetic mode)'

settings = dict(mode=MODE, collection=COLLECTION_ID, requested_start=START_UTC,
    requested_end=END_UTC, region=REGION_NAME if MODE == 'real' else 'artificial ellipse',
    observation_start=times[0], observation_end=times[-1], lookback=LOOKBACK,
    step_minutes=STEP_MINUTES, horizon_minutes=HORIZON_MINUTES, input_size=INPUT_SIZE,
    fine_size=FINE_SIZE, projection=PROJECTION, extent=EXTENT, radius_m=RADIUS_M,
    min_target_coverage=MIN_TARGET_COVERAGE, min_input_coverage=MIN_INPUT_COVERAGE,
    splits=partitions, rejected_samples=rejected, train_share=TRAIN_SHARE, validation_share=VALID_SHARE,
    batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, epochs_limit=EPOCHS,
    patience=PATIENCE, seed=SEED, best_epoch=best_epoch, best_validation_mse=best_val,
    training_target_mean=training_mean, device=str(DEVICE), versions=versions)
write_json(RUN / 'run_config.json', settings)
write_json(RUN / 'package_versions.json', versions)

# Reconstruct the same architecture, load trusted weights, and predict one retained example.
reloaded = CloudCNNLSTM().to(DEVICE)
reloaded.load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True))
reloaded.eval()
one_image_sequence, one_clock_sequence, _ = datasets['test'][0]
with torch.no_grad():
    one_forecast = reloaded(one_image_sequence.unsqueeze(0).to(DEVICE),
                            one_clock_sequence.unsqueeze(0).to(DEVICE)).item()
assert np.isclose(one_forecast, network_predictions[0], atol=1e-6)
print(f'Checkpoint reload verified. First forecast: {100 * one_forecast:.1f}% cloud cover')
print('Saved outputs:', RUN)
print('\n'.join(sorted(path.name for path in RUN.iterdir())))


## 11. What to discuss in your assignment

- State the response, geography, grid, history length, lead time, dates, and quality thresholds. Include the boundary and first-frame plots to establish that the target is meaningful.
- Report retained sample counts and exclusions, chronological split dates, validation-selected epoch, and all three test MAE/RMSE values. Explain whether the CNN–LSTM improves on persistence, including negative results.
- Satellite CLM is the measurement target, not perfect ground truth. Classification errors, coastlines, projection/resampling, missing data, and delivery latency limit interpretation.
- A two-week seasonal pilot cannot establish year-round skill. Extend to multiple seasons and keep a genuinely later test period. Tune history length and image size using validation only.
- Do not submit synthetic results as satellite forecasts. The smoke test checks the software interface, not EUMETSAT authentication, real geolocation, or forecast skill.

### Troubleshooting

| Symptom | Action |
|---|---|
| Authentication/403 error | Verify the consumer key/secret, account access and collection terms; do not print the token. |
| No catalogue products | Check dates and collection availability; keep the requested CLM collection. |
| ecCodes DLL/library error | Install the native ecCodes stack through conda-forge in the selected kernel. |
| Satpy cannot identify a file | Inspect its original name and format against the reader configuration; keep the original GRIB filename. |
| Unexpected categories or almost no valid Irish pixels | Stop; inspect reader metadata, grid alignment, missing values, and the decoded preview. Do not relax QC to hide a decoding problem. |
| Too few samples | Inspect the saved frame times/coverage and extend the continuous date range. Gaps and partition boundaries intentionally discard examples. |
| Slow preprocessing | Reuse raw downloads and caches. Native full-disk decoding remains expensive even for a small output crop. |
| Out of memory | Reduce the date range or use a disk-backed frame dataset; lower batch size for training. |

### Primary references

- [EUMETSAT CLM collection](https://api.eumetsat.int/data/browse/1.0.0/collections/EO%3AEUM%3ADAT%3AMSG%3ACLM?format=html): requested data product.
- [EUMETSAT cloud-mask code table](https://vocabulary-manager.eumetsat.int/vocabularies/GRIB/WMO/100/TABLE_CODE_FLAG/4.217): category meanings and missing code.
- [EUMDAC guide](https://user.eumetsat.int/resources/user-guides/eumetsat-data-access-client-eumdac-guide) and [official Data Store tutorials](https://gitlab.eumetsat.int/eumetlab/data-services/eumdac_data_store): access and downloads.
- [Satpy reader configuration](https://github.com/pytroll/satpy/blob/main/satpy/etc/readers/seviri_l2_grib.yaml) and [GRIB reader documentation](https://satpy.readthedocs.io/en/stable/api/satpy.readers.eum_l2_grib.html): original file matching and `cloud_mask` support.
- [Satpy resampling](https://satpy.readthedocs.io/en/stable/resample.html): putting scenes on fixed grids.
- [Natural Earth countries](https://www.naturalearthdata.com/downloads/10m-cultural-vectors/10m-admin-0-countries/): generalized default boundary (public domain).
- [PyTorch LSTM](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html): recurrent-layer interface.
